## Jacobi iteration

Ax = b

x(0)_guess = [0, 0, 0]

A matrix

D matrix = diagonal matrix

L matrix = (-1) * lower triangular matrix

U matrix = (-1) * upper triangular matrix

M = D
N = L + U
Tj = D^(-1) * (L + U)

in each iteration: we multiply Tj with the previous x(k) and add it to b

x(k+1) = D^(-1) * (L+U) * x(k) + D^(-1) * b = M^(-1) * N * x(k) + M^(-1) * b

stop condition: do this until the difference between x(k+1) and x(k) is less than the tolerance

x = [x1, x2, x3] (column vector)

||x||_infinity = max(|x1|, |x2|, |x3|)

jacob_it(A, b, x0, err, max_iterations)
x0 = [0, 0, 0] -> initial guess

In [17]:
import numpy as np


def inf_norm(x: np.ndarray) -> float:
    """Infinity (Chebyshev) norm for vectors: max_i |x_i|."""
    return float(np.max(np.abs(x)))


def matrix_inf_norm(A: np.ndarray) -> float:
    """Infinity norm for matrices: max row sum of absolute values."""
    return float(np.max(np.sum(np.abs(A), axis=1)))


In [18]:
import numpy as np


def jacobi_it(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray,
    err: float,
    max_iterations: int,
) -> tuple[np.ndarray, int, list[float]]:
    """
    Jacobi iteration for solving A x = b.
    Stop when ||x^(k+1) - x^(k)||_inf < err or when max_iterations is reached.
    """
    A = A.astype(float)
    b = b.astype(float)
    x = x0.astype(float).copy()

    D = np.diag(np.diag(A))
    # Match the course notation: L and U are minus the triangular parts of A.
    L = -np.tril(A, k=-1)
    U = -np.triu(A, k=1)

    M = D
    N = L + U

    if np.any(np.isclose(np.diag(M), 0.0)):
        raise ValueError("Jacobi method cannot be applied: A has zero on diagonal.")

    history: list[float] = []

    for k in range(1, max_iterations + 1):
        # x^(k+1) = M^(-1) N x^(k) + M^(-1) b
        x_new = np.linalg.solve(M, N @ x + b)

        diff_inf = inf_norm(x_new - x)
        history.append(diff_inf)

        if diff_inf < err:
            return x_new, k, history

        x = x_new

    return x, max_iterations, history


# Example
A = np.array([[5, -1, 0], [-1, 5, -1], [0, -1, 5]], dtype=float)
b = np.array([4, 3, 4], dtype=float)
x0 = np.zeros_like(b)

a, iters, diffs = jacobi_it(A, b, x0, err=0.000001, max_iterations=1000)
print("Approximate solution:", a)
print("Iterations:", iters)
print("Last infinity-norm difference:", diffs[-1])



Approximate solution: [0.99999995 0.9999999  0.99999995]
Iterations: 13
Last infinity-norm difference: 2.0971520009460676e-07


## Gauss-Seidel iteration

M = D - La
N = U

everithing else is the same as in Jacobi iteration

In [19]:
def gauss_seidel_it(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray,
    err: float,
    max_iterations: int,
) -> tuple[np.ndarray, int, list[float]]:
    """
    Gauss-Seidel iteration for solving A x = b.
    Stop when ||x^(k+1) - x^(k)||_inf < err or when max_iterations is reached.
    """
    A = A.astype(float)
    b = b.astype(float)
    x = x0.astype(float).copy()

    D = np.diag(np.diag(A))
    # Match the course notation: L and U are minus the triangular parts of A.
    L = -np.tril(A, k=-1)
    U = -np.triu(A, k=1)

    M = D - L
    N = U

    if np.any(np.isclose(np.diag(M), 0.0)):
        raise ValueError("Gauss-Seidel method cannot be applied: A has zero on diagonal.")

    history: list[float] = []

    for k in range(1, max_iterations + 1):
        # x^(k+1) = M^(-1) N x^(k) + M^(-1) b
        x_new = np.linalg.solve(M, N @ x + b)

        diff_inf = inf_norm(x_new - x)
        history.append(diff_inf)

        if diff_inf < err:
            return x_new, k, history

        x = x_new

    return x, max_iterations, history


# Same input as Jacobi example above
x_gs, iters_gs, diffs_gs = gauss_seidel_it(A, b, x0, err=0.000001, max_iterations=1000)
print("Gauss-Seidel approximate solution:", x_gs)
print("Gauss-Seidel iterations:", iters_gs)
print("Gauss-Seidel last infinity-norm difference:", diffs_gs[-1])

Gauss-Seidel approximate solution: [0.99999999 0.99999999 1.        ]
Gauss-Seidel iterations: 8
Gauss-Seidel last infinity-norm difference: 1.447034881918441e-07


Ax = b -> x
Ax = b_tilda -> x_tilda

compare x and x_tilda

|| x - x_tilda || / || x || -> output relative error

|| b - b_tilda || / || b || -> input relative error

|| A - A_tilda || / || A || -> input relative error




In [22]:
def errors_when_perturbing_b(
    A: np.ndarray,
    b: np.ndarray,
    b_tilde: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    """
    Use original b and directly provided b_tilde.
    Returns x, x_tilde, output_relative_error, input_relative_error.
    """
    A = A.astype(float)
    b = b.astype(float)
    b_tilde = b_tilde.astype(float)

    x = np.linalg.solve(A, b)
    x_tilde = np.linalg.solve(A, b_tilde)

    output_rel_error = inf_norm(x - x_tilde) / inf_norm(x)
    input_rel_error = inf_norm(b - b_tilde) / inf_norm(b)

    return x, x_tilde, output_rel_error, input_rel_error


def errors_when_perturbing_A(
    A: np.ndarray,
    b: np.ndarray,
    A_tilde: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    """
    Use original A and directly provided A_tilde.
    Returns x, x_tilde, output_relative_error, input_relative_error.
    """
    A = A.astype(float)
    b = b.astype(float)
    A_tilde = A_tilde.astype(float)

    x = np.linalg.solve(A, b)
    x_tilde = np.linalg.solve(A_tilde, b)

    output_rel_error = inf_norm(x - x_tilde) / inf_norm(x)
    input_rel_error = matrix_inf_norm(A - A_tilde) / matrix_inf_norm(A)

    return x, x_tilde, output_rel_error, input_rel_error


# Example using the same system as above
A = np.array([[10, 7, 8, 7], [7, 5, 6, 5], [8, 6, 10, 9], [7, 5, 9, 10]], dtype=float)
b = np.array([32, 23, 33, 31], dtype=float)

b_tilde = np.array([32.1, 22.9, 33.1, 30.9], dtype=float)
A_tilde = np.array([[10, 7, 8.1, 7.2], [7.8, 5.04, 6, 5], [8, 5.98, 9.89, 9], [6.99, 4.99, 9, 9.98]], dtype=float)

x, x_tilde_b, out_err_b, in_err_b = errors_when_perturbing_b(A, b, b_tilde)
print("Perturb b:")
print("x =", x)
print("x_tilde =", x_tilde_b)
print("output relative error =", out_err_b)
print("input relative error  =", in_err_b)

x, x_tilde_A, out_err_A, in_err_A = errors_when_perturbing_A(A, b, A_tilde)
print("\nPerturb A:")
print("x =", x)
print("x_tilde =", x_tilde_A)
print("output relative error =", out_err_A)
print("input relative error  =", in_err_A)

print()
print("Condition number of A:", np.linalg.cond(A))

Perturb b:
x = [1. 1. 1. 1.]
x_tilde = [  9.2 -12.6   4.5  -1.1]
output relative error = 13.600000000000007
input relative error  = 0.0030303030303030732

Perturb A:
x = [1. 1. 1. 1.]
x_tilde = [0.05702583 2.36523548 0.81566576 1.14808342]
output relative error = 1.3652354833762308
input relative error  = 0.025454545454545452

Condition number of A: 2984.092701675935


## Interpretation of the observed phenomenon

The experiment shows a strong **sensitivity to perturbations**:

- For perturbing `b`, the input relative error is very small (`~0.003`), but the output relative error is huge (`~13.6`).
- For perturbing `A`, the input relative error is still small (`~0.025`), yet the output relative error is also very large (`~1.365`).

This means the linear system is **ill-conditioned**: small changes in data (`A` or `b`) can produce large changes in the solution `x`.

An ill-conditioned system is a set of linear equations highly sensitive to small perturbations, where minor changes in input data (or rounding errors) result in massive fluctuations in the output solution. 

These systems have a high condition number, indicating instability, often occurring when rows/columns are nearly linearly dependent, behaving almost like dividing by zero. 